In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

"""
DATA PREPROCESSING PIPELINE
---------------------------
Concept: This script demonstrates a standard data preprocessing workflow.
It handles missing values, encodes categorical variables, and strictly
prevents data leakage by splitting the data first.
*Note: StandardScaler is temporarily disabled to view raw imputed values.*
"""

# 1. Create a mock dataset for demonstration
data = {
    'Age': [25, 30, np.nan, 45, 22, 50, 35],
    'Salary': [50000, 64000, 58000, np.nan, 45000, 120000, 70000],
    'Department': ['IT', 'HR', 'IT', 'Marketing', 'HR', 'Marketing', 'IT'],
    'Purchased': ['Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes'] # Target variable
}
df = pd.DataFrame(data)

print("--- Original Raw Data ---")
print(df, "\n")

# 2. Separate Features (X) from the Target (y)
# Why: The model uses X to predict y. We do not preprocess the target variable
# the exact same way we preprocess the input features.
X = df.drop('Purchased', axis=1)
y = df['Purchased']

# 3. Train-Test Split (CRITICAL STEP)
# Why: We split BEFORE applying any transformations to prevent data leakage.
# The model should never "see" information from the test set during training.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Identify column types for targeted preprocessing
# Why: Numerical and categorical data require entirely different mathematical treatments.
numeric_features = ['Age', 'Salary']
categorical_features = ['Department']

# 5. Build the Preprocessing Pipelines
# Why: Pipelines chain steps together cleanly. If we impute missing values,
# we must do that BEFORE we try to scale the numbers.

# Numeric Pipeline: Fill missing values with the median ONLY (scaling removed for checking)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
    # ('scaler', StandardScaler()) # <-- Commented out so you can read the normal ages
])

# Categorical Pipeline: Fill missing values with the most frequent, then One-Hot Encode
# handle_unknown='ignore' ensures the model doesn't crash if it sees a new category in production
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 6. Combine pipelines using ColumnTransformer
# Why: ColumnTransformer applies the specific pipelines to the correct columns simultaneously.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 7. Execute the Preprocessing
# Fit AND Transform the training data (learn the medians, means, and categories, then apply them)
X_train_processed = preprocessor.fit_transform(X_train)

# ONLY Transform the testing data (use the medians/means learned ONLY from the training data)
X_test_processed = preprocessor.transform(X_test)

# 8. View the results (Optional: Convert back to DataFrame for readability)
# Get the new column names generated by the One-Hot Encoder
encoded_cat_cols = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
all_cols = numeric_features + list(encoded_cat_cols)

X_train_final = pd.DataFrame(X_train_processed, columns=all_cols)

print("--- Fully Preprocessed Training Data (Unscaled) ---")
print(X_train_final.round(3))

--- Original Raw Data ---
    Age    Salary Department Purchased
0  25.0   50000.0         IT       Yes
1  30.0   64000.0         HR        No
2   NaN   58000.0         IT        No
3  45.0       NaN  Marketing       Yes
4  22.0   45000.0         HR        No
5  50.0  120000.0  Marketing       Yes
6  35.0   70000.0         IT       Yes 

--- Fully Preprocessed Training Data (Unscaled) ---
    Age    Salary  Department_HR  Department_IT  Department_Marketing
0  50.0  120000.0            0.0            0.0                   1.0
1  40.0   58000.0            0.0            1.0                   0.0
2  22.0   45000.0            1.0            0.0                   0.0
3  45.0   64000.0            0.0            0.0                   1.0
4  35.0   70000.0            0.0            1.0                   0.0
